# ncm-forge — end-to-end pilot on a free Colab T4

Trains the two stages of [Next-Concept Modeling](https://arxiv.org/abs/2610.14162)
on a slice of JavaScript: a residual **VQ-VAE tokenizer**, then a **135 M
generator** that fills code in the middle (FIM), and prints a completion.

**This is a smoke test, not a usable model.** 800 tokenizer steps and 300
generator steps on ~20 M tokens give a model that outputs mostly noise. It shows
that the pipeline runs end to end, reports real numbers, and takes ~35 min on a
free T4.

**What this notebook will not do**: produce a useful autocomplete. On a 4070 the
same pipeline needs 10 h and 210 M tokens for a model that is still weak
(see `docs/BENCH.md`, "capabilities').

The dataset is gated: accept the terms on
https://huggingface.co/datasets/bigcode/starcoderdata first.

*Generated by `scripts/build_notebook.py` — do not edit this file by hand.*

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 1. Get the code and the upstream model modules

In [ ]:
!git clone -q https://github.com/Bakhaow/ncm-forge.git
%cd ncm-forge
!pip install -q torch transformers omegaconf numpy
!git clone --depth 1 -q https://github.com/HazyResearch/ncm.git vendor/ncm

## 2. Accept the dataset gate and log in (one paste, no token in the notebook)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 3. Build a ~20 M token JavaScript corpus from one parquet shard

In [ ]:
from huggingface_hub import hf_hub_download
path = hf_hub_download(
    "bigcode/starcoderdata",
    "javascript/train-00000-of-00065.parquet",
    repo_type="dataset",
    local_dir="data/raw/starcoderdata",
)
print(path)

In [ ]:
!python scripts/build_js_corpus.py --parquet data/raw/starcoderdata/javascript/*.parquet \
    --out-dir data/js --context-lines 30 --max-tokens 20000000

## 4. Shrink the config for a T4 and this corpus

In [ ]:
import pathlib
s = pathlib.Path("configs/code_js.yaml").read_text()
s = s.replace("corpus: data/js_full/fim.jsonl", "corpus: data/js/fim.jsonl")
s = s.replace("steps: 14190", "steps: 300")
s = s.replace("checkpoint_path: out/pilot/tokenizer/tokenizer.pt",
              "checkpoint_path: out/colab/tokenizer/tokenizer.pt")
s = s.replace("tokenizer_out_dir: out/js/tokenizer", "tokenizer_out_dir: out/colab/tokenizer")
s = s.replace("out_dir: out/js/generator", "out_dir: out/colab/generator")
s = s.replace("measured_gen_it_s: 0.52", "measured_gen_it_s: 0.10")
pathlib.Path("configs/colab.yaml").write_text(s)
print("configs/colab.yaml written")

## 5. Stage 1 — the VQ-VAE tokenizer (800 steps)

Watch `token_accuracy`: it is the share of tokens the tokenizer reconstructs
exactly. It is the ceiling of everything downstream.

In [ ]:
!python -m forge.cli train-tokenizer --config configs/colab.yaml \
    data.max_docs=12000 train.steps=800 train.log_every=100

## 6. Stage 2 — the generator (300 steps)

`loss` is the per-level cross entropy. On 300 steps it barely moves; the point is
the plumbing, not the number.

In [ ]:
!python -m forge.cli train-generator --config configs/colab.yaml \
    train.log_every=50

## 7. Fill a function in (the interesting artefact)

The model generates one level at a time, coarse to fine. `--show-levels` prints
that trajectory: level 1 is the shape, the last level the text.

In [ ]:
from pathlib import Path
Path("data/example.js").write_text('''import { readFile } from "node:fs/promises";

async function loadConfig(path) {
  const raw = await readFile(path, "utf8");
''')
!python scripts/generate.py --config configs/colab.yaml \
    --tokenizer out/colab/tokenizer/tokenizer.pt \
    --generator out/colab/generator/generator.pt \
    --file data/example.js --show-levels

## 8. Where to go from here

- **Real run**: 210 M tokens, 14 190 steps, ~7.6 h at batch 32 x 8 accumulations
  on a 4070 — `configs/js_final.yaml`. Measured throughput, OOM points and the
  context-length ablation are in `docs/BENCH.md`.
- **Your own domain**: `scripts/build_fim_dataset.py --src <tree>` builds FIM
  windows from any source tree, then copy `configs/code_js.yaml`.
- **Sanity check before believing a number**: `tests/test_pipeline.py` pins the
  level offsets and the FIM alignment; `scripts/ab_context.py` cross-evaluates two
  checkpoints at several context lengths.

A 300-step drafter that produces plausible JavaScript would be a bug in the
evaluation, not a result.